In [53]:
import rich
import logging
import glob

import matplotlib.pyplot as plt
import time
import os
import awkward as ak
import numpy as np
import pandas as pd
from lgdo import lh5
from legendmeta import LegendMetadata
from dbetto import Props
from pygama.pargen.AoE_cal import *
from pygama.pargen.AoE_cal import CalAoE, Pol1, SigmaFit, aoe_peak
from pygama.pargen.data_cleaning import get_tcm_pulser_ids
from pygama.pargen.utils import load_data
from tqdm import tqdm
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages

import os, json
import pint

u = pint.get_application_registry()


from dspeed.vis.waveform_browser import WaveformBrowser
from legendmeta import LegendMetadata
from dbetto import Props
from pygama.pargen.utils import load_data

%matplotlib inline


# INizio con il run 000

In [54]:
data_path = "/global/cfs/cdirs/m2676/data/lngs/l200/public/prodenv/prod-blind/ref/latest"
config = Props.read_from(os.path.join(data_path, "config.json"), subst_pathvar=True)["setups"]["l200"]["paths"]
meta  = LegendMetadata(config["metadata"])
chmap = meta.channelmap(meta.dataprod.runinfo.p03.r000.phy.start_key)

In [55]:
data_path

'/global/cfs/cdirs/m2676/data/lngs/l200/public/prodenv/prod-blind/ref/latest'

In [56]:
# PRENDO I DATI DI FISICA E NON DI CALIBRAZIONE f"{config["tier_psp"]}/cal/p03/r000/*.lh5" ---> f"{config["tier_psp"]}/phy/p03/r000/*.lh5"
pet_file = sorted(glob.glob(f"{config["tier_pet"]}/phy/l200-p03-r000-phy-tier_pet.lh5"))
psp_file = sorted(glob.glob(f"{config["tier_psp"]}/phy/p03/r000/*.lh5")) #access exsiting DSP-processed data such as the energy for partitions
tcm_file = sorted(glob.glob(f"{config["tier_tcm"]}/phy/p03/r000/*.lh5"))  # --> solo per il pulse
raw_file = sorted(glob.glob(f"{config["tier_raw"]}/phy/p03/r000/*.lh5"))

# basic

In [57]:
browser = WaveformBrowser(raw_file, f"ch{chanmap.V02160A.daq.rawid}/raw", lines="waveform_presummed", n_drawn=10)
browser.draw_next()
browser.new_figure()
browser.draw_next()

NameError: name 'chanmap' is not defined

# cuts

In [ ]:
plt.rcParams["figure.figsize"] = (14, 4)
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["font.size"] = 14

In [ ]:
channel = f'ch{chmap.V02160A.daq.rawid}'
detector = "V02160A"

In [ ]:
cal_file = f"{config["par_pht"]}/cal/p03/r000/l200-p03-r000-cal-20230311T235840Z-par_pht.json"
cal_dict = Props.read_from(cal_file)[channel]["pars"]["operations"]

In [ ]:
params = [ "is_valid_bl_slope_classifier",
         "cuspEmax_ctc_cal"]

In [ ]:
# qui mi posso salvare i paramteri di DSP (che ho scelto e definito in config) per lo specfico canale e per tutti gli eventi registrati
start = time.time()
data = load_data(
        files = psp_file,   # da qui prenidmao i dati di dsp già processati 
        lh5_path = f"{channel}/dsp",  # prendo i risultati del dsp per il canale /detcetor specifico
        cal_dict = Props.read_from(cal_file)[channel]["pars"]["operations"],  # lo prendiamo da par_pht
        params=params,
        return_selection_mask=False,)

print(f"[INFO] Took {time.time() - start:.2f} s")


In [ ]:
# trigger
trigger_data = lh5.read_as("/evt/trigger", pet_file, library="ak")
data["forced_trigger"] = ak.to_numpy(trigger_data['is_forced'])

# pulsesr and coincidences 
coincident_data = lh5.read_as("/evt/coincident", pet_file, library="ak")
data["is_pulser"] = coincident_data['puls']  # no pulser events
data["is_muon"] = coincident_data['muon_offline']  # no muons
data["is_HPGe"] = coincident_data['geds']  # only events with an HPGe signal

# energy selection
mask_lowe =  (data["cuspEmax_ctc_cal"] >=0 ) & (data["cuspEmax_ctc_cal"] < 500)
data["is_below_500keV"] = mask_lowe

### Discharge
geds_data = lh5.read_as("/evt/geds", pet_file, library="ak")
mask_discharge = geds_data.quality.is_not_bb_like.is_delayed_discharge
data["is_discharge"] = mask_discharge



In [ ]:
m_fix = data["is_HPGe"]&~data["is_muon"]
m_discharge = ~data["is_discharge"]
m_bl_slope_above5 = (data["is_valid_bl_slope_classifier"] > 5)
    
m_pulser = data["is_pulser"]
m_ft = data["forced_trigger"]
m_lowE = data["is_below_500keV"]

In [ ]:
Low_energy_selection =  m_discharge & ~m_pulser & ~m_ft & m_fix & m_lowE & m_bl_slope_above5 
print("voglio vedere vere quali eventi vengono rigettati dai tagli")
print(np.sum(Low_energy_selection), "Low energy events")

In [ ]:
meta.dataprod.runinfo.p03

In [ ]:
dsp_config = meta.dataprod.config.tier_dsp['l200-p01-r%-T%-ICPC-dsp_proc_chain']
print("Available processors:", list(dsp_config['processors'].keys()))
dsp_overrides = meta.dataprod.overrides.dsp.on(meta.dataprod.runinfo.p03.r000.cal.start_key).ch1104000

In [ ]:
print(list(meta.dataprod.config.tier_dsp.keys()))

In [ ]:
browser = WaveformBrowser(
    raw_file,
    f"{channel}/raw",
    dsp_config=dsp_config,  # Need to include a dsp config file!
    database=dsp_overrides,
    lines=["wf_blsub"],
    aux_values=data,   # dsp data
    #lines=["waveform_presummed"],
    legend=["E = {cuspEmax_ctc_cal} keV"],
    #legend_opts={"loc": "upper left"},
    #legend=["A/E = {AoE_Classifier}"],  # values to put in the legend
    entry_mask=Low_energy_selection,  # apply cut
    n_drawn=3,  # number to draw for draw_next
)

browser.draw_next()

plt.ylabel("Signal (ADC)")
plt.xlabel("Time (s)")
plt.grid(alpha = 0.25, ls = '--')

# All the detectors run 000

# BL SLOPE

In [ ]:
cal = Props.read_from(cal_file)

In [ ]:
# tra tutti i detector che ci sono prendo solo quelli con lo stato di 
# usability = 'on' oppure 'ac'
# processability = True

chmap = meta.channelmap(meta.dataprod.runinfo.p03.r000.phy.start_key)
geds = chmap.group("system").geds

chmap_ak = ak.Array(geds.values())

rawids = ak.to_numpy(chmap_ak.daq.rawid)
names = ak.to_numpy(chmap_ak["name"]).astype(str)

# Leggi la usability di ciascun detector
usability = np.array([
    getattr(det.analysis, "usability", None)
    for det in geds.values()
])

# Tieni tutto tranne i detector con usability == "off"
mask_not_off = (usability != "off")

# (Opzionale) Controlla quali stati sono presenti
print("Usability presenti:", np.unique(usability))

# Maschere per tipo di detector, escludendo solo gli "off"
masks_name = {
    "ICPC": np.char.startswith(names, "V") & mask_not_off,
    "BEGe": np.char.startswith(names, "B") & mask_not_off,
    "Coax": np.char.startswith(names, "C") & mask_not_off,
    "PPC":  np.char.startswith(names, "P") & mask_not_off,
}
det_names = ["ICPC", "BEGe", "Coax", "PPC"]

In [58]:
channels_rawid = {
    det_type: [f"ch{rid}" for rid in rawids[mask]]
    for det_type, mask in masks_name.items()
}

channels_names = {
    det_type: names[mask].tolist()
    for det_type, mask in masks_name.items()
}

print("\nExcluded channels (usability='off' OR processable=False):")

for ch, det in geds.items():

    usability = getattr(det.analysis, "usability", None)
    processable = getattr(det.analysis, "processable", None)

    if (usability == "off") or (processable is False):

        print(
            f"channel={ch:3d} | "
            f"rawid=ch{det.daq.rawid:5d} | "
            f"name={det.name:20s} | "
            f"usability={usability} | "
            f"processable={processable}"
        )

# Riassunto
for det_type in channels_names:
    print(f"\n{det_type}: {len(channels_names[det_type])} detector")
    print(channels_names[det_type])


Excluded channels (usability='off' OR processable=False):
channel= 17 | rawid=ch1108804 | name=V07298B              | usability=off | processable=False
channel= 21 | rawid=ch1110405 | name=P00665A              | usability=off | processable=False
channel= 34 | rawid=ch1115200 | name=V01386A              | usability=off | processable=False
channel= 35 | rawid=ch1115201 | name=V01403A              | usability=off | processable=False
channel= 36 | rawid=ch1115202 | name=V01404A              | usability=off | processable=False
channel= 64 | rawid=ch1080002 | name=B00091D              | usability=off | processable=False
channel= 65 | rawid=ch1080003 | name=P00537A              | usability=off | processable=False
channel= 74 | rawid=ch1083200 | name=B00091B              | usability=off | processable=False
channel= 91 | rawid=ch1088000 | name=P00538B              | usability=off | processable=False
channel= 95 | rawid=ch1088004 | name=P00661A              | usability=off | processable=False
c

In [59]:
cal_file = f"{config["par_pht"]}/cal/p03/r000/l200-p03-r000-cal-20230311T235840Z-par_pht.json"
trigger_data = lh5.read_as("/evt/trigger", pet_file, library="ak")
coincident_data = lh5.read_as("/evt/coincident", pet_file, library="ak")

In [62]:
dsp_config = meta.dataprod.config.tier_dsp[
    'l200-p01-r%-T%-ICPC-dsp_proc_chain'
]

detector_type = "ICPC"
output_pdf = "waveforms_ICPC.pdf"
params = [
    "is_valid_bl_slope_classifier",
    "is_valid_bl_slope",
    "cuspEmax_ctc_cal"
]

# --------------------------
# PRE-CACHING (fuori loop)
# --------------------------
start_key = meta.dataprod.runinfo.p03.r000.cal.start_key
dsp_overrides_all = meta.dataprod.overrides.dsp.on(start_key)

forced_trigger_np = ak.to_numpy(trigger_data["is_forced"])

channels_list = channels_rawid[detector_type]

with PdfPages(output_pdf) as pdf:
    for i, channel in tqdm(enumerate(channels_list)):
        print(f"\n==================================== {channel} {i+1}/{len(channels_list)} ====================================\n")
    
        # --------------------------
        # CALIBRAZIONE
        # --------------------------
        cal_dict = Props.read_from(cal_file)[channel]["pars"]["operations"]
    
        # --------------------------
        # LOAD DATA (NO COPY)
        # --------------------------
        data = load_data(
            files=psp_file,
            lh5_path=f"{channel}/dsp",
            cal_dict=cal_dict,
            params=params,
            return_selection_mask=False,
        )
    
        # --------------------------
        # FLAGS GLOBALI
        # --------------------------
        data["forced_trigger"] = forced_trigger_np
        data["is_pulser"] = coincident_data["puls"]
        data["is_muon"] = coincident_data["muon_offline"]
        data["is_HPGe"] = coincident_data["geds"]
    
        data["is_discharge"] = geds_data.quality.is_not_bb_like.is_delayed_discharge
    
        data["is_below_250keV"] = (
            (data["cuspEmax_ctc_cal"] > 0) &
            (data["cuspEmax_ctc_cal"] < 250)
        )
    
        # --------------------------
        # CACHE LOCALI (IMPORTANTE)
        # --------------------------
        is_hpge = data["is_HPGe"]
        is_muon = data["is_muon"]
        is_pulser = data["is_pulser"]
        forced_trigger = data["forced_trigger"]
        is_discharge = data["is_discharge"]
        is_below_250keV = data["is_below_250keV"]
        bl_slope = data["is_valid_bl_slope_classifier"]
    
        # --------------------------
        # MASCHERE
        # --------------------------
        m_fix = is_hpge & ~is_muon
        m_discharge = ~is_discharge
        m_bl_slope = bl_slope > 5
    
        # --------------------------
        # SELEZIONE
        # --------------------------
        low_energy_selection = (
            m_discharge &
            ~is_pulser &
            ~forced_trigger &
            m_fix &
            is_below_250keV &
            m_bl_slope
        )
    
        n_selected = int(np.sum(low_energy_selection))
        print(n_selected, "Low energy events")
    
        if n_selected == 0:
            print("No events → skipping channel")
            continue
    
        # --------------------------
        # OVERRIDES DSP
        # --------------------------
        dsp_overrides = dsp_overrides_all[channel]
    
        # --------------------------
        # BROWSER
        # --------------------------
        # colormap fredda
        cmap = plt.cm.Blues  # alternativa: plt.cm.Greys, plt.cm.cividis
        
        n = n_selected
        colors = cmap(np.linspace(0.3, 0.95, n))
        browser = WaveformBrowser(
            raw_file,
            f"{channel}/raw",
            dsp_config=dsp_config,
            database=dsp_overrides,
            lines=["wf_blsub"],
            aux_values=data,
            legend=["E = {cuspEmax_ctc_cal} keV"],
            entry_mask=low_energy_selection,
            n_drawn=n_selected,
            styles={"color": colors}
        )
    
        browser.draw_next()
    
        plt.title(f"chn : {channel}")
        plt.ylabel("Signal (ADC)")
        plt.xlabel("Time (ns)")
        plt.grid(alpha=0.25, ls="--")

        fig = plt.gcf()   # prende la figura corrente

        pdf.savefig(fig)
        plt.close(fig)

0it [00:00, ?it/s]


==================================== ch1104000 1/37 ====================================

3 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
1it [00:41, 41.76s/it]


==================================== ch1104001 2/37 ====================================

3 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
2it [01:00, 28.47s/it]

...

/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
35it [10:41, 19.74s/it]


==================================== ch1086400 36/37 ====================================

73 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
36it [11:00, 19.30s/it]


==================================== ch1086401 37/37 ====================================

23 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
37it [11:14, 18.24s/it]


In [118]:
dsp_config = meta.dataprod.config.tier_dsp[
    'l200-p01-r%-T%-ICPC-dsp_proc_chain'
]
output_pdf = "waveforms_bege.pdf"
detector_type = "BEGe"
params = [
    "is_valid_bl_slope_classifier",
    "is_valid_bl_slope",
    "cuspEmax_ctc_cal"
]

# --------------------------
# PRE-CACHING (fuori loop)
# --------------------------
start_key = meta.dataprod.runinfo.p03.r000.cal.start_key
dsp_overrides_all = meta.dataprod.overrides.dsp.on(start_key)

forced_trigger_np = ak.to_numpy(trigger_data["is_forced"])

channels_list = channels_rawid[detector_type]

with PdfPages(output_pdf) as pdf:
    for i, channel in tqdm(enumerate(channels_list)):
        print(f"\n==================================== {channel} {i+1}/{len(channels_list)} ====================================\n")
    
        # --------------------------
        # CALIBRAZIONE
        # --------------------------
        cal_dict = Props.read_from(cal_file)[channel]["pars"]["operations"]
    
        # --------------------------
        # LOAD DATA (NO COPY)
        # --------------------------
        data = load_data(
            files=psp_file,
            lh5_path=f"{channel}/dsp",
            cal_dict=cal_dict,
            params=params,
            return_selection_mask=False,
        )
    
        # --------------------------
        # FLAGS GLOBALI
        # --------------------------
        data["forced_trigger"] = forced_trigger_np
        data["is_pulser"] = coincident_data["puls"]
        data["is_muon"] = coincident_data["muon_offline"]
        data["is_HPGe"] = coincident_data["geds"]
    
        data["is_discharge"] = geds_data.quality.is_not_bb_like.is_delayed_discharge
    
        data["is_below_250keV"] = (
            (data["cuspEmax_ctc_cal"] > 0) &
            (data["cuspEmax_ctc_cal"] < 250)
        )
    
        # --------------------------
        # CACHE LOCALI (IMPORTANTE)
        # --------------------------
        is_hpge = data["is_HPGe"]
        is_muon = data["is_muon"]
        is_pulser = data["is_pulser"]
        forced_trigger = data["forced_trigger"]
        is_discharge = data["is_discharge"]
        is_below_250keV = data["is_below_250keV"]
        bl_slope = data["is_valid_bl_slope_classifier"]
    
        # --------------------------
        # MASCHERE
        # --------------------------
        m_fix = is_hpge & ~is_muon
        m_discharge = ~is_discharge
        m_bl_slope = bl_slope > 5
    
        # --------------------------
        # SELEZIONE
        # --------------------------
        low_energy_selection = (
            m_discharge &
            ~is_pulser &
            ~forced_trigger &
            m_fix &
            is_below_250keV &
            m_bl_slope
        )
    
        n_selected = int(np.sum(low_energy_selection))
        print(n_selected, "Low energy events")
    
        if n_selected == 0:
            print("No events → skipping channel")
            continue
    
        # --------------------------
        # OVERRIDES DSP
        # --------------------------
        dsp_overrides = dsp_overrides_all[channel]
    
        # --------------------------
        # BROWSER
        # --------------------------
        # colormap fredda
        cmap = plt.cm.Blues  # alternativa: plt.cm.Greys, plt.cm.cividis
        
        n = n_selected
        colors = cmap(np.linspace(0.3, 0.95, n))
        browser = WaveformBrowser(
            raw_file,
            f"{channel}/raw",
            dsp_config=dsp_config,
            database=dsp_overrides,
            lines=["wf_blsub"],
            aux_values=data,
            legend=["E = {cuspEmax_ctc_cal} keV"],
            entry_mask=low_energy_selection,
            n_drawn=n_selected,
            styles={"color": colors}
        )
    
        browser.draw_next()
    
        plt.title(f"chn : {channel}")
        plt.ylabel("Signal (ADC)")
        plt.xlabel("Time (s)")
        plt.grid(alpha=0.25, ls="--")
    
        fig = plt.gcf()   # prende la figura corrente

        pdf.savefig(fig)
        plt.close(fig)

0it [00:00, ?it/s]


==================================== ch1107202 1/26 ====================================

1 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
1it [00:20, 20.70s/it]


==================================== ch1110402 2/26 ====================================

3 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
2it [00:37, 18.37s/it]

...


==================================== ch1086404 25/26 ====================================

3 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
25it [09:02, 20.58s/it]


==================================== ch1086405 26/26 ====================================



26it [09:15, 21.38s/it]

0 Low energy events
No events → skipping channel


In [117]:
import os
os.getcwd()

'/global/u2/r/ritaferi'

In [119]:
dsp_config = meta.dataprod.config.tier_dsp[
    'l200-p01-r%-T%-ICPC-dsp_proc_chain'
]
output_pdf = "waveforms_ppc.pdf"
detector_type = "PPC"
params = [
    "is_valid_bl_slope_classifier",
    "is_valid_bl_slope",
    "cuspEmax_ctc_cal"
]

# --------------------------
# PRE-CACHING (fuori loop)
# --------------------------
start_key = meta.dataprod.runinfo.p03.r000.cal.start_key
dsp_overrides_all = meta.dataprod.overrides.dsp.on(start_key)

forced_trigger_np = ak.to_numpy(trigger_data["is_forced"])

channels_list = channels_rawid[detector_type]

with PdfPages(output_pdf) as pdf:
    for i, channel in tqdm(enumerate(channels_list)):
        print(f"\n==================================== {channel} {i+1}/{len(channels_list)} ====================================\n")
    
        # --------------------------
        # CALIBRAZIONE
        # --------------------------
        cal_dict = Props.read_from(cal_file)[channel]["pars"]["operations"]
    
        # --------------------------
        # LOAD DATA (NO COPY)
        # --------------------------
        data = load_data(
            files=psp_file,
            lh5_path=f"{channel}/dsp",
            cal_dict=cal_dict,
            params=params,
            return_selection_mask=False,
        )
    
        # --------------------------
        # FLAGS GLOBALI
        # --------------------------
        data["forced_trigger"] = forced_trigger_np
        data["is_pulser"] = coincident_data["puls"]
        data["is_muon"] = coincident_data["muon_offline"]
        data["is_HPGe"] = coincident_data["geds"]
    
        data["is_discharge"] = geds_data.quality.is_not_bb_like.is_delayed_discharge
    
        data["is_below_250keV"] = (
            (data["cuspEmax_ctc_cal"] > 0) &
            (data["cuspEmax_ctc_cal"] < 250)
        )
    
        # --------------------------
        # CACHE LOCALI (IMPORTANTE)
        # --------------------------
        is_hpge = data["is_HPGe"]
        is_muon = data["is_muon"]
        is_pulser = data["is_pulser"]
        forced_trigger = data["forced_trigger"]
        is_discharge = data["is_discharge"]
        is_below_250keV = data["is_below_250keV"]
        bl_slope = data["is_valid_bl_slope_classifier"]
    
        # --------------------------
        # MASCHERE
        # --------------------------
        m_fix = is_hpge & ~is_muon
        m_discharge = ~is_discharge
        m_bl_slope = bl_slope > 5
    
        # --------------------------
        # SELEZIONE
        # --------------------------
        low_energy_selection = (
            m_discharge &
            ~is_pulser &
            ~forced_trigger &
            m_fix &
            is_below_250keV &
            m_bl_slope
        )
    
        n_selected = int(np.sum(low_energy_selection))
        print(n_selected, "Low energy events")
    
        if n_selected == 0:
            print("No events → skipping channel")
            continue
    
        # --------------------------
        # OVERRIDES DSP
        # --------------------------
        dsp_overrides = dsp_overrides_all[channel]
    
        # --------------------------
        # BROWSER
        # --------------------------
        # colormap fredda
        cmap = plt.cm.Blues  # alternativa: plt.cm.Greys, plt.cm.cividis
        
        n = n_selected
        colors = cmap(np.linspace(0.3, 0.95, n))
        browser = WaveformBrowser(
            raw_file,
            f"{channel}/raw",
            dsp_config=dsp_config,
            database=dsp_overrides,
            lines=["wf_blsub"],
            aux_values=data,
            legend=["E = {cuspEmax_ctc_cal} keV"],
            entry_mask=low_energy_selection,
            n_drawn=n_selected,
            styles={"color": colors}
        )
    
        browser.draw_next()
    
        plt.title(f"chn : {channel}")
        plt.ylabel("Signal (ADC)")
        plt.xlabel("Time (s)")
        plt.grid(alpha=0.25, ls="--")
    
        fig = plt.gcf()   # prende la figura corrente

        pdf.savefig(fig)
        plt.close(fig)

0it [00:00, ?it/s]


==================================== ch1110404 1/20 ====================================

4 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
1it [00:41, 41.89s/it]


==================================== ch1112000 2/20 ====================================

2 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
2it [01:01, 28.76s/it]

...

/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
18it [06:30, 19.49s/it]


==================================== ch1089601 19/20 ====================================

2 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
19it [06:49, 19.33s/it]


==================================== ch1089603 20/20 ====================================

6 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
20it [07:07, 21.39s/it]


In [123]:

dsp_config = meta.dataprod.config.tier_dsp[
    'l200-p01-r%-T%-COAX-dsp_proc_chain'
]
print("Available processors:", list(dsp_config['processors'].keys()))
dsp_overrides = meta.dataprod.overrides.dsp.on(meta.dataprod.runinfo.p03.r000.cal.start_key).ch1104000

Available processors: ['tp_interp', 'tp10_interp', 'tp50_interp', 'tp90_interp', 'tp_ann_levels', 'tp_ann_0', 'tp_ann', 'rt10_90']


In [124]:
# coax


dsp_config = meta.dataprod.config.tier_dsp[
    'l200-p01-r%-T%-ICPC-dsp_proc_chain'
]
output_pdf = "waveforms_coax.pdf"
detector_type = "Coax"
params = [
    "is_valid_bl_slope_classifier",
    "is_valid_bl_slope",
    "cuspEmax_ctc_cal"
]

# --------------------------
# PRE-CACHING (fuori loop)
# --------------------------
start_key = meta.dataprod.runinfo.p03.r000.cal.start_key
dsp_overrides_all = meta.dataprod.overrides.dsp.on(start_key)

forced_trigger_np = ak.to_numpy(trigger_data["is_forced"])

channels_list = channels_rawid[detector_type]

with PdfPages(output_pdf) as pdf:
    for i, channel in tqdm(enumerate(channels_list)):
        print(f"\n==================================== {channel} {i+1}/{len(channels_list)} ====================================\n")
    
        # --------------------------
        # CALIBRAZIONE
        # --------------------------
        cal_dict = Props.read_from(cal_file)[channel]["pars"]["operations"]
    
        # --------------------------
        # LOAD DATA (NO COPY)
        # --------------------------
        data = load_data(
            files=psp_file,
            lh5_path=f"{channel}/dsp",
            cal_dict=cal_dict,
            params=params,
            return_selection_mask=False,
        )
    
        # --------------------------
        # FLAGS GLOBALI
        # --------------------------
        data["forced_trigger"] = forced_trigger_np
        data["is_pulser"] = coincident_data["puls"]
        data["is_muon"] = coincident_data["muon_offline"]
        data["is_HPGe"] = coincident_data["geds"]
    
        data["is_discharge"] = geds_data.quality.is_not_bb_like.is_delayed_discharge
    
        data["is_below_250keV"] = (
            (data["cuspEmax_ctc_cal"] > 0) &
            (data["cuspEmax_ctc_cal"] < 250)
        )
    
        # --------------------------
        # CACHE LOCALI (IMPORTANTE)
        # --------------------------
        is_hpge = data["is_HPGe"]
        is_muon = data["is_muon"]
        is_pulser = data["is_pulser"]
        forced_trigger = data["forced_trigger"]
        is_discharge = data["is_discharge"]
        is_below_250keV = data["is_below_250keV"]
        bl_slope = data["is_valid_bl_slope_classifier"]
    
        # --------------------------
        # MASCHERE
        # --------------------------
        m_fix = is_hpge & ~is_muon
        m_discharge = ~is_discharge
        m_bl_slope = bl_slope > 5
    
        # --------------------------
        # SELEZIONE
        # --------------------------
        low_energy_selection = (
            m_discharge &
            ~is_pulser &
            ~forced_trigger &
            m_fix &
            is_below_250keV &
            m_bl_slope
        )
    
        n_selected = int(np.sum(low_energy_selection))
        print(n_selected, "Low energy events")
    
        if n_selected == 0:
            print("No events → skipping channel")
            continue
    
        # --------------------------
        # OVERRIDES DSP
        # --------------------------
        dsp_overrides = dsp_overrides_all[channel]
    
        # --------------------------
        # BROWSER
        # --------------------------
        # colormap fredda
        cmap = plt.cm.Blues  # alternativa: plt.cm.Greys, plt.cm.cividis
        
        n = n_selected
        colors = cmap(np.linspace(0.3, 0.95, n))
        browser = WaveformBrowser(
            raw_file,
            f"{channel}/raw",
            dsp_config=dsp_config,
            database=dsp_overrides,
            lines=["wf_blsub"],
            aux_values=data,
            legend=["E = {cuspEmax_ctc_cal} keV"],
            entry_mask=low_energy_selection,
            n_drawn=n_selected,
            styles={"color": colors}
        )
    
        browser.draw_next()
    
        plt.title(f"chn : {channel}")
        plt.ylabel("Signal (ADC)")
        plt.xlabel("Time (s)")
        plt.grid(alpha=0.25, ls="--")
    
        fig = plt.gcf()   # prende la figura corrente

        pdf.savefig(fig)
        plt.close(fig)

0it [00:00, ?it/s]


==================================== ch1107203 1/6 ====================================

1 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
1it [00:50, 50.54s/it]


==================================== ch1107204 2/6 ====================================

31 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
2it [01:10, 32.44s/it]

...

/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
4it [01:55, 26.10s/it]


==================================== ch1108801 5/6 ====================================

3 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
5it [02:18, 24.66s/it]


==================================== ch1120005 6/6 ====================================

106 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
6it [03:09, 31.50s/it]


# bl_slope_RMS

In [32]:
channels_tot = []

for ch, det in geds.items():

    usability = getattr(det.analysis, "usability", None)
    processable = getattr(det.analysis, "processable", None)

    if usability in ("on", "ac"):
        channels_tot.append(f"ch{det.daq.rawid}")

In [138]:
# coax


dsp_config = meta.dataprod.config.tier_dsp[
    'l200-p01-r%-T%-ICPC-dsp_proc_chain'
]
output_pdf = "waveforms_bl_slope_RMS.pdf"
params = [
    #"is_valid_bl_slope_classifier",
    "is_valid_bl_slope",
    "is_valid_bl_slope_rms_classifier",
    "is_valid_bl_slope_rms",
    "cuspEmax_ctc_cal"
]

# --------------------------
# PRE-CACHING (fuori loop)
# --------------------------
start_key = meta.dataprod.runinfo.p03.r000.cal.start_key
dsp_overrides_all = meta.dataprod.overrides.dsp.on(start_key)

forced_trigger_np = ak.to_numpy(trigger_data["is_forced"])
channels_list = channels_tot
with PdfPages(output_pdf) as pdf:
    for i, channel in tqdm(enumerate(channels_list)):
        print(f"\n==================================== {channel} {i+1}/{len(channels_list)} ====================================\n")
    
        # --------------------------
        # CALIBRAZIONE
        # --------------------------
        cal_dict = Props.read_from(cal_file)[channel]["pars"]["operations"]
    
        # --------------------------
        # LOAD DATA (NO COPY)
        # --------------------------
        data = load_data(
            files=psp_file,
            lh5_path=f"{channel}/dsp",
            cal_dict=cal_dict,
            params=params,
            return_selection_mask=False,
        )
    
        # --------------------------
        # FLAGS GLOBALI
        # --------------------------
        data["forced_trigger"] = forced_trigger_np
        data["is_pulser"] = coincident_data["puls"]
        data["is_muon"] = coincident_data["muon_offline"]
        data["is_HPGe"] = coincident_data["geds"]
    
        data["is_discharge"] = geds_data.quality.is_not_bb_like.is_delayed_discharge
    
        data["is_below_250keV"] = (
            (data["cuspEmax_ctc_cal"] > 0) &
            (data["cuspEmax_ctc_cal"] < 250)
        )
    
        # --------------------------
        # CACHE LOCALI (IMPORTANTE)
        # --------------------------
        is_hpge = data["is_HPGe"]
        is_muon = data["is_muon"]
        is_pulser = data["is_pulser"]
        forced_trigger = data["forced_trigger"]
        is_discharge = data["is_discharge"]
        is_below_250keV = data["is_below_250keV"]
        bl_slope = data["is_valid_bl_slope"]

        bl_slope_rms_class = data["is_valid_bl_slope_rms_classifier"]
    
        # --------------------------
        # MASCHERE
        # --------------------------
        m_fix = is_hpge & ~is_muon
        m_discharge = ~is_discharge
        m_qc = bl_slope 

        m_bl_slope_rms = bl_slope_rms_class > 5  # taglio per guardare gli eventi che verrebbero rigettati dal taglio in analisi
    
        # --------------------------
        # SELEZIONE
        # --------------------------
        low_energy_selection = (
            m_discharge &
            ~is_pulser &
            ~forced_trigger &
            m_fix &
            is_below_250keV &
            m_qc &
            m_bl_slope_rms
        )
    
        n_selected = int(np.sum(low_energy_selection))
        print(n_selected, "Low energy events")
    
        if n_selected == 0:
            print("No events → skipping channel")
            continue
    
        # --------------------------
        # OVERRIDES DSP
        # --------------------------
        dsp_overrides = dsp_overrides_all[channel]
    
        # --------------------------
        # BROWSER
        # --------------------------
        # colormap fredda
        cmap = plt.cm.Blues  # alternativa: plt.cm.Greys, plt.cm.cividis
        
        n = n_selected
        colors = cmap(np.linspace(0.3, 0.95, n))
        browser = WaveformBrowser(
            raw_file,
            f"{channel}/raw",
            dsp_config=dsp_config,
            database=dsp_overrides,
            lines=["wf_blsub"],
            aux_values=data,
            legend=["E = {cuspEmax_ctc_cal} keV"],
            legend_opts={"loc": "lower right"},
            entry_mask=low_energy_selection,
            n_drawn=n_selected,
            styles={"color": colors}
        )
    
        browser.draw_next()
        plt.title(f"chn : {channel}")
        plt.ylabel("Signal (ADC)")
        plt.xlabel("Time (s)")
        plt.grid(alpha=0.25, ls="--")
    
        fig = plt.gcf()   # prende la figura corrente

        pdf.savefig(fig)
        plt.close(fig)

print("=======================  finished ===========================")

0it [00:00, ?it/s]


==================================== ch1104000 1/89 ====================================

1 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
1it [00:17, 17.82s/it]


==================================== ch1104001 2/89 ====================================

7 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
2it [00:37, 18.94s/it]

...

0 Low energy events
No events → skipping channel

==================================== ch1089601 88/89 ====================================



88it [26:14, 16.23s/it]

0 Low energy events
No events → skipping channel

==================================== ch1089603 89/89 ====================================

25 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
89it [26:37, 17.95s/it]

=======================  finished ===========================


# taill rms

In [34]:
channels_tot;

In [35]:
dsp_config = meta.dataprod.config.tier_dsp[
    'l200-p01-r%-T%-ICPC-dsp_proc_chain'
]
output_pdf = "waveforms_tail_RMS.pdf"
params = [
    #"is_valid_bl_slope_classifier",
    "is_valid_bl_slope",
    #"is_valid_bl_slope_rms_classifier",
    "is_valid_bl_slope_rms",
    "is_valid_tail_rms_classifier",
    "is_valid_tail_rms",
    "cuspEmax_ctc_cal"
]
#channels_list = channels_rawid[detector_type]

# --------------------------
# PRE-CACHING (fuori loop)
# --------------------------
start_key = meta.dataprod.runinfo.p03.r000.cal.start_key
dsp_overrides_all = meta.dataprod.overrides.dsp.on(start_key)

forced_trigger_np = ak.to_numpy(trigger_data["is_forced"])
channels_list = channels_tot
with PdfPages(output_pdf) as pdf:
    for i, channel in tqdm(enumerate(channels_list)):
        print(f"\n==================================== {channel} {i+1}/{len(channels_list)} ====================================\n")
    
        # --------------------------
        # CALIBRAZIONE
        # --------------------------
        cal_dict = Props.read_from(cal_file)[channel]["pars"]["operations"]
    
        # --------------------------
        # LOAD DATA (NO COPY)
        # --------------------------
        data = load_data(
            files=psp_file,
            lh5_path=f"{channel}/dsp",
            cal_dict=cal_dict,
            params=params,
            return_selection_mask=False,
        )
    
        # --------------------------
        # FLAGS GLOBALI
        # --------------------------
        data["forced_trigger"] = forced_trigger_np
        data["is_pulser"] = coincident_data["puls"]
        data["is_muon"] = coincident_data["muon_offline"]
        data["is_HPGe"] = coincident_data["geds"]
    
        data["is_discharge"] = geds_data.quality.is_not_bb_like.is_delayed_discharge
    
        data["is_below_250keV"] = (
            (data["cuspEmax_ctc_cal"] > 0) &
            (data["cuspEmax_ctc_cal"] < 250)
        )
    
        # --------------------------
        # CACHE LOCALI (IMPORTANTE)
        # --------------------------
        is_hpge = data["is_HPGe"]
        is_muon = data["is_muon"]
        is_pulser = data["is_pulser"]
        forced_trigger = data["forced_trigger"]
        is_discharge = data["is_discharge"]
        is_below_250keV = data["is_below_250keV"]
        bl_slope = data["is_valid_bl_slope"]
        bl_slope_rms_class = data["is_valid_bl_slope_rms"]

        tail_rms_class = data["is_valid_tail_rms_classifier"]
    
        # --------------------------
        # MASCHERE
        # --------------------------
        m_fix = is_hpge & ~is_muon
        m_discharge = ~is_discharge
        m_qc = bl_slope & bl_slope_rms_class

        m_tail_rms = tail_rms_class > 5  # taglio per guardare gli eventi che verrebbero rigettati dal taglio in analisi
    
        # --------------------------
        # SELEZIONE
        # --------------------------
        low_energy_selection = (
            m_discharge &
            ~is_pulser &
            ~forced_trigger &
            m_fix &
            is_below_250keV &
            m_qc &
            m_tail_rms
        )
    
        n_selected = int(np.sum(low_energy_selection))
        print(n_selected, "Low energy events")
    
        if n_selected == 0:
            print("No events → skipping channel")
            continue
    
        # --------------------------
        # OVERRIDES DSP
        # --------------------------
        dsp_overrides = dsp_overrides_all[channel]
    
        # --------------------------
        # BROWSER
        # --------------------------
        # colormap fredda
        cmap = plt.cm.Blues  # alternativa: plt.cm.Greys, plt.cm.cividis
        
        n = n_selected
        colors = cmap(np.linspace(0.3, 0.95, n))
        browser = WaveformBrowser(
            raw_file,
            f"{channel}/raw",
            dsp_config=dsp_config,
            database=dsp_overrides,
            lines=[#"wf_blsub",
                    "wf_pz"],
            aux_values=data,
            legend=["E = {cuspEmax_ctc_cal} keV"],
            legend_opts={"loc": "lower right"},
            entry_mask=low_energy_selection,
            n_drawn=n_selected,
            styles={"color": colors}
        )
    
        browser.draw_next()
        plt.title(f"chn : {channel}")
        plt.ylabel("Signal (ADC)")
        plt.xlabel("Time (ns)")
        plt.grid(alpha=0.25, ls="--")
    
        fig = plt.gcf()   # prende la figura corrente

        pdf.savefig(fig)
        plt.close(fig)

print("=======================  finished ===========================")

0it [00:00, ?it/s]


==================================== ch1104000 1/89 ====================================

3 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
1it [00:36, 36.70s/it]


==================================== ch1104001 2/89 ====================================

10 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
2it [00:48, 21.96s/it]

...


==================================== ch1089601 88/89 ====================================

1 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
88it [18:24, 13.15s/it]


==================================== ch1089603 89/89 ====================================

4 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
89it [18:37, 12.56s/it]

=======================  finished ===========================


# trap TpMAX

In [52]:
meta.dataprod.config.tier_dsp

TextDB('/global/cfs/cdirs/m2676/data/lngs/l200/public/prodenv/prod-blind/ref/v2.1.5/inputs/dataprod/config/tier_dsp')

In [51]:
dsp_config = meta.dataprod.config.tier_dsp[
    'l200-p01-r%-T%-ICPC-dsp_proc_chain'
]
output_pdf = "waveforms_tp_TrapMax_passed.pdf"
params = [
    "is_valid_bl_slope",
    "is_valid_bl_slope_rms",
    "is_valid_tail_rms",
    "is_not_noise_burst",
    "is_valid_cuspEmin",
    "is_valid_cuspEmax",
    "tp_trapTmax",
    "cuspEmax_ctc_cal"
]
#channels_list = channels_rawid[detector_type]

# --------------------------
# PRE-CACHING (fuori loop)
# --------------------------
start_key = meta.dataprod.runinfo.p03.r000.cal.start_key
dsp_overrides_all = meta.dataprod.overrides.dsp.on(start_key)

forced_trigger_np = ak.to_numpy(trigger_data["is_forced"])
channels_list = channels_tot
with PdfPages(output_pdf) as pdf:
    for i, channel in tqdm(enumerate(channels_list)):
        print(f"\n==================================== {channel} {i+1}/{len(channels_list)} ====================================\n")
    
        # --------------------------
        # CALIBRAZIONE
        # --------------------------
        cal_dict = Props.read_from(cal_file)[channel]["pars"]["operations"]
    
        # --------------------------
        # LOAD DATA (NO COPY)
        # --------------------------
        data = load_data(
            files=psp_file,
            lh5_path=f"{channel}/dsp",
            cal_dict=cal_dict,
            params=params,
            return_selection_mask=False,
        )
    
        # --------------------------
        # FLAGS GLOBALI
        # --------------------------
        data["forced_trigger"] = forced_trigger_np
        data["is_pulser"] = coincident_data["puls"]
        data["is_muon"] = coincident_data["muon_offline"]
        data["is_HPGe"] = coincident_data["geds"]
    
        data["is_discharge"] = geds_data.quality.is_not_bb_like.is_delayed_discharge
    
        data["is_below_100keV"] = (
            (data["cuspEmax_ctc_cal"] > 0) &
            (data["cuspEmax_ctc_cal"] < 40)
        )
    
        # --------------------------
        # CACHE LOCALI (IMPORTANTE)
        # --------------------------
        is_hpge = data["is_HPGe"]
        is_muon = data["is_muon"]
        is_pulser = data["is_pulser"]
        forced_trigger = data["forced_trigger"]
        is_discharge = data["is_discharge"]
        is_below_100keV = data["is_below_100keV"]
        bl_slope = data["is_valid_bl_slope"]
        bl_slope_rms_class = data["is_valid_bl_slope_rms"]
        tail_rms_class = data["is_valid_tail_rms"]
        noise_burst = data["is_not_noise_burst"]
        cuspEmin = data["is_valid_cuspEmin"]
        cuspEmax = data["is_valid_cuspEmax"]
    
        # --------------------------
        # MASCHERE
        # --------------------------
        m_fix = is_hpge & ~is_muon
        m_discharge = ~is_discharge
        m_qc = bl_slope & bl_slope_rms_class & tail_rms_class & noise_burst & cuspEmin & ~cuspEmax

        m_failed_trapTmax = (data["tp_trapTmax"] > 55000 ) & ( data["tp_trapTmax"] < 65000)
    
        # --------------------------
        # SELEZIONE
        # --------------------------
        low_energy_selection = (
            m_discharge &
            ~is_pulser &
            ~forced_trigger &
            m_fix &
            is_below_100keV &
            m_qc &
            m_failed_trapTmax
        )
    
        n_selected = int(np.sum(low_energy_selection))
        print(n_selected, "Low energy events")
    
        if n_selected == 0:
            print("No events → skipping channel")
            continue
    
        # --------------------------
        # OVERRIDES DSP
        # --------------------------
        dsp_overrides = dsp_overrides_all[channel]
    
        # --------------------------
        # BROWSER
        # --------------------------
        # colormap fredda
        cmap = plt.cm.Blues  # alternativa: plt.cm.Greys, plt.cm.cividis
        
        n = n_selected
        colors = cmap(np.linspace(0.3, 0.95, n))
        browser = WaveformBrowser(
            raw_file,
            f"{channel}/raw",
            dsp_config=dsp_config,
            database=dsp_overrides,
            lines=[#"wf_blsub",
                    "wf_pz",],
                    #"wf_trap",],
                    #"cuspEmax",],
            aux_values=data,
            legend=["E = {cuspEmax_ctc_cal} keV"],
            legend_opts={"loc": "lower right"},
            entry_mask=low_energy_selection,
            n_drawn=n_selected,
            styles={"color": colors}
            
        )
    
        browser.draw_next()
        plt.title(f"chn : {channel}")
        plt.ylabel("Signal (ADC)")
        plt.xlabel("Time (ns)")
        plt.grid(alpha=0.25, ls="--")
    
        fig = plt.gcf()   # prende la figura corrente

        pdf.savefig(fig)
        plt.close(fig)

print("=======================  finished ===========================")

0it [00:00, ?it/s]


==================================== ch1104000 1/89 ====================================

18 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
1it [00:18, 18.56s/it]


==================================== ch1104001 2/89 ====================================

40 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
2it [00:36, 18.01s/it]

...


==================================== ch1089601 88/89 ====================================

5 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
88it [23:04, 13.29s/it]


==================================== ch1089603 89/89 ====================================

9 Low energy events


/opt/conda/lib/python3.12/site-packages/dspeed/processing_chain.py:1571: RuntimeWarning: invalid value encountered in divide
  self.processor(*self.args, **self.kwargs)
89it [23:17, 15.70s/it]

=======================  finished ===========================
